In [ ]:
# Load all scraped data into the 3-table schema
# Sources, Articles, Framing_Analysis

import json
import csv

# ── SOURCES TABLE ──────────────────────────────────────────────────────────
sources = [
    {
        "Source_ID": "the guardian",
        "Website_Name": "The Guardian",
        "Base_URL": "https://www.theguardian.com",
        "Scraping_Permission": "Allowed"
    },
    {
        "Source_ID": "Wikipedia",
        "Website_Name": "Wikipedia",
        "Base_URL": "https://en.wikipedia.org",
        "Scraping_Permission": "Allowed"
    }
]

with open('sources.json', 'w') as f:
    json.dump(sources, f, indent=4)

print(f'Sources table: {len(sources)} entries saved to sources.json')

In [ ]:
# ── ARTICLES TABLE ─────────────────────────────────────────────────────────
# NOTE: Wikipedia data has no Article_Text yet (scraped as 'Pending Phase 3')
# and most entries are navigation links, not real articles.
# Only the 4 Guardian articles are fully usable right now.
# Once you run your other scrapers, append those results here too.

import json

articles = []

# Load Guardian articles (already have Article_ID)
with open('../Guardian_Scraper/Articles.json', 'r', encoding='utf-8') as f:
    guardian_articles = json.load(f)

for article in guardian_articles:
    articles.append({
        "Article_ID": article['Article_ID'],
        "Source_ID": article['Source_ID'],
        "Title": article['Title'],
        "Publish_Date": article['Publish_Date'],
        "URL": article['URL'],
        "Article_Text": article['Article_Text']
    })

# Load Wikipedia articles — filter out junk entries (short titles, no real content)
with open('../selenium_articles_data.json', 'r', encoding='utf-8') as f:
    wiki_articles = json.load(f)

wiki_added = 0
for index, article in enumerate(wiki_articles, start=1):
    # Skip navigation links and entries with no real content
    if len(article['Title']) < 20:
        continue
    if article['Article_Text'] == 'Pending Phase 3':
        continue

    articles.append({
        "Article_ID": f'W{index}',
        "Source_ID": article['Source_ID'],
        "Title": article['Title'],
        "Publish_Date": article['Publish_Date'],
        "URL": article['URL'],
        "Article_Text": article['Article_Text']
    })
    wiki_added += 1

with open('articles_table.json', 'w', encoding='utf-8') as f:
    json.dump(articles, f, indent=4, ensure_ascii=False)

print(f'Guardian articles loaded: {len(guardian_articles)}')
print(f'Wikipedia articles loaded: {wiki_added} (rest were junk navigation links)')
print(f'Total articles table: {len(articles)} entries saved to articles_table.json')

In [ ]:
# ── FRAMING_ANALYSIS TABLE ─────────────────────────────────────────────────

import json

framing_analysis = []

# Load existing Guardian framing results from Source_ID.json
with open('../Guardian_Scraper/Source_ID.json', 'r', encoding='utf-8') as f:
    guardian_framing = json.load(f)

for index, entry in enumerate(guardian_framing, start=1):
    framing_analysis.append({
        "Analysis_ID": f'A{index}',
        "Article_ID": entry['Article_ID'],
        "Target_Nation": entry['Target_Nation'],
        "Frequency_of_Political_Terms": entry['Frequency_of_Political_Terms'],
        "Stance_Score": None  # to be filled in Phase 3
    })

with open('framing_analysis_table.json', 'w', encoding='utf-8') as f:
    json.dump(framing_analysis, f, indent=4)

print(f'Framing analysis table: {len(framing_analysis)} entries saved to framing_analysis_table.json')
print()
for entry in framing_analysis:
    print(entry)

In [ ]:
# ── EXPORT ALL 3 TABLES TO CSV ─────────────────────────────────────────────

import json
import csv

def save_csv(filename, data, columns):
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(data)
    print(f'Saved {filename}')

with open('sources.json') as f:
    sources = json.load(f)
with open('articles_table.json') as f:
    articles = json.load(f)
with open('framing_analysis_table.json') as f:
    framing = json.load(f)

save_csv('sources.csv', sources,
         ['Source_ID', 'Website_Name', 'Base_URL', 'Scraping_Permission'])

save_csv('articles.csv', articles,
         ['Article_ID', 'Source_ID', 'Title', 'Publish_Date', 'URL', 'Article_Text'])

save_csv('framing_analysis.csv', framing,
         ['Analysis_ID', 'Article_ID', 'Target_Nation', 'Frequency_of_Political_Terms', 'Stance_Score'])

print()
print('All 3 tables exported. Summary:')
print(f'  sources.csv          — {len(sources)} rows')
print(f'  articles.csv         — {len(articles)} rows')
print(f'  framing_analysis.csv — {len(framing)} rows')